# Gráfico combinado: variação das matrículas × escolas com variação brusca

Gera **um único gráfico** (três painéis empilhados: anos iniciais, anos finais e ensino médio) que sobrepõe, em cada painel:

- **barras (eixo esquerdo):** variação anual (%) da matrícula total de cada rede em relação ao ano anterior;
- **linhas (eixo direito):** % de escolas de cada rede com **queda brusca** (tracejada) ou **aumento brusco** (contínua), pela cerca de Tukey;
- **faixas cinza:** anos da pandemia (`ANOS_PANDEMIA`).

**Atenção ao universo de cada informação:** as barras usam **todas** as escolas da rede; as linhas usam só as escolas abertas nos dois anos e com
pelo menos `MATRICULA_MINIMA_ANO_ANTERIOR` alunos no ano anterior (o mesmo critério do notebook de variações bruscas).
Os dois eixos têm unidades diferentes (variação % da matrícula × % de escolas); por isso o gráfico tem dois eixos Y.

**Como usar:** edite só a célula da **Etapa 1 (parâmetros)** — tudo que é cor, fonte, tamanho e texto está lá — e rode as células em ordem.

## Etapa 0 — Imports

In [35]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

try:
    display
except NameError:                       # rodando fora do Jupyter
    def display(x):
        print(x)

# Para salvar imagens é preciso:  %pip install vl-convert-python   (e reiniciar o kernel)

## Etapa 1 — PARÂMETROS (edite só aqui)

In [ ]:
# ============================================================
#  PARÂMETROS — TUDO QUE VOCÊ PRECISA EDITAR ESTÁ NESTA CÉLULA
# ============================================================

# ---------- 1) ARQUIVO ----------
RAIZ = Path.cwd()
while not (RAIZ / "dados").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
CAMINHO_ARQUIVO = RAIZ / "dados" / "bd5.zip"   # .zip (com um .csv dentro) ou direto .csv
NOME_ARQUIVO_NO_ZIP = None    # se o zip tiver vários arquivos, escreva o nome. None = usa o primeiro .csv
CSV_SEP      = ";"
CSV_ENCODING = "utf-8"        # se der erro de acentos, tente "latin-1"

# ---------- 2) O QUE ANALISAR ----------
ETAPAS_PAINEIS   = ["AI", "AF", "EM"]   # um painel por item, de cima para baixo. Aceita somas: "AI+AF", e "TOTAL"
INCLUIR_INTEGRAL = True       # True = regular + integral | False = só regular
DIMENSAO         = "REDE"     # "REDE" (Pública × Privada), "DEP_ADM", "LOCALIZACAO", "REGIAO" ou "UF"
ANO_INICIO = 2012             # primeiro ano com variação (2011 só serve de base)
ANO_FIM    = 2025

# ---------- 3) FILTROS (None = não filtra) ----------
FILTRO_REDE        = None     # ex.: ["Pública"]
FILTRO_DEP_ADM     = None     # ex.: ["Privada", "Municipal"]
FILTRO_LOCALIZACAO = None     # ex.: ["Urbana"]
FILTRO_REGIAO      = None     # ex.: ["Sudeste", "Sul"]
FILTRO_UF          = None     # ex.: ["RJ", "SP"]

# ---------- 4) CRITÉRIO DE VARIAÇÃO BRUSCA (as linhas) ----------
Q_INFERIOR    = 0.25          # quartil de baixo (0.25 = Q1)
Q_SUPERIOR    = 0.75          # quartil de cima  (0.75 = Q3)
MULTIPLICADOR = 1.5           # o "k" da cerca de Tukey: Q1 - k*AIQ e Q3 + k*AIQ
CRITERIO = "estatistico"      # "estatistico" | "absoluto" | "ambos"
CORTE_ABSOLUTO_PCT         = -30   # queda: variação <= este valor (%), se CRITERIO = "absoluto" ou "ambos"
CORTE_ABSOLUTO_AUMENTO_PCT = 30    # aumento: variação >= este valor (%)
LIMITE_PARA_CRUZAMENTO = "ANO"     # "ANO" = cercas calculadas dentro de cada ano | "GLOBAL" = todos os anos juntos
MATRICULA_MINIMA_ANO_ANTERIOR = 30 # ignora escolas muito pequenas (ruído)
EXCLUIR_FECHAMENTO = True     # True = tira escolas com 0 matrículas no ano atual
N_MIN_CATEGORIA = 30          # categorias com menos escolas/ano que isso não entram nas linhas

# ---------- 5) FAIXA DOS ANOS DA PANDEMIA ----------
ANOS_PANDEMIA     = [2019, 2020, 2021, 2022]
COR_FAIXA         = "#9CA3AF"
OPACIDADE_FAIXA   = 0.25
GRADE_ANOS        = True      # True = tracejado vertical separando os anos
COR_GRADE_ANOS    = "#9CA3AF"
TRACO_GRADE_ANOS  = [4, 4]    # [traço, espaço]

# ---------- 6) CORES ----------
COR_GRUPOS = {                # chave = valor original dos dados (em português). Grupo fora da lista usa ESQUEMA_CORES
    "Privada": "#7B2CBF", "Pública": "#1FAB04",
    "Federal": "#1B9E77", "Estadual": "#D95F02", "Municipal": "#E7298A",
    "Urbana": "#7B2CBF", "Rural": "#F4A100",
}
ESQUEMA_CORES     = "set2"    # usado se algum grupo não estiver em COR_GRUPOS
COR_ZERO          = "#374151" # linha do zero das barras
COR_TEXTO         = "#111827" # títulos
COR_TEXTO_SUAVE   = "#374151" # rótulos dos eixos
COR_GRADE_Y       = "#E5E7EB" # grade horizontal
COR_FUNDO         = "white"

# ---------- 7) BARRAS E LINHAS ----------
OPACIDADE_BARRAS  = 0.55      # 1 = sólidas; menor deixa as linhas mais visíveis por cima
LARGURA_LINHA     = 2.8
TAMANHO_PONTO     = 70
TRACEJADO_QUEDA   = [7, 4]    # padrão da linha de queda brusca (a de aumento é contínua)
LINHAS_SOBRE_BARRAS = True    # True = as linhas ficam por cima das barras

# ---------- 8) EIXOS ----------
EIXO_Y_COMPARTILHADO = True   # True = mesma escala nos três painéis (comparação justa)
Y_BARRAS = None               # None = automático (simétrico). Ou fixe, ex.: (-8, 8)
Y_LINHAS = None               # None = automático (começa em 0). Ou fixe, ex.: (0, 12)
ALINHAR_ZEROS = False         # True = o zero do eixo direito fica na mesma altura do zero das barras

# ---------- 9) TAMANHOS E FONTES ----------
LARGURA_GRAFICO     = 1000    # largura de cada painel (px)
ALTURA_PAINEL       = 400     # altura de cada painel (px)
ESPACO_PAINEIS      = 22      # espaço vertical entre painéis (px)
FONTE_TITULO        = 32      # título principal
FONTE_SUBTITULO     = 20      # subtítulo
FONTE_TITULO_PAINEL = 20      # título de cada painel (etapa)
FONTE_EIXOS         = 15      # números dos eixos
FONTE_EIXOS_TITULO  = 17      # títulos dos eixos
FONTE_LEGENDA       = 17      # itens da legenda
FONTE_LEGENDA_TITULO = 14     # títulos da legenda
POSICAO_LEGENDA     = "top" # "right", "left", "top", "bottom"

# ---------- 10) TEXTOS (em inglês; edite à vontade) ----------
TITULO = ["Annual enrollment change and share of schools",
          "with sharp changes, by network and stage"]
SUBTITULO = ["Bars: annual change in total enrollment (left axis).",
             "Lines: % of schools with a sharp drop (dashed) or sharp increase (solid) (right axis).",
             "Shaded: 2019–2022."]
TEXTO_EIXO_X       = "Year"
TEXTO_EIXO_BARRAS  = "Annual change in enrollment (%)"
TEXTO_EIXO_LINHAS  = "% of schools with a sharp change"
TEXTO_LEGENDA_GRUPO = "Network"
TEXTO_LEGENDA_TIPO  = "Share of schools"
ROTULO_QUEDA   = "Sharp drop"
ROTULO_AUMENTO = "Sharp increase"
ROTULOS_GRUPOS = {            # como cada categoria aparece no gráfico
    "Pública": "Public", "Privada": "Private", "Federal": "Federal", "Estadual": "State", "Municipal": "Municipal",
    "Urbana": "Urban", "Rural": "Rural",
}
ROTULOS_ETAPAS = {            # nome de cada painel
    "AI": "Early elementary (AI)", "AF": "Late elementary (AF)", "EM": "High school (EM)",
    "TOTAL": "Total (AI + AF + EM)",
}

# ---------- 11) SALVAR A IMAGEM ----------
SALVAR_IMAGEM   = True       # True = salva em PASTA_IMAGENS (precisa de vl-convert-python)
FORMATOS_IMAGEM = ["pdf"]     # "pdf", "png" e/ou "svg"
ESCALA_PNG      = 3           # resolução do PNG (3 = ~300 dpi)
PASTA_IMAGENS   = RAIZ / "analise" / "resultados"
NOME_IMAGEM     = "grafico_combinado"

## Etapa 2 — Carregar a base e calcular as duas informações

Para cada etapa de `ETAPAS_PAINEIS`:

1. **barras:** soma a matrícula de todas as escolas de cada grupo em cada ano e calcula a variação % em relação ao ano anterior;
2. **linhas:** calcula a variação % de cada escola (anos consecutivos, base mínima), as cercas de Tukey e, em cada grupo e ano,
   a % de escolas com queda brusca e com aumento brusco.

In [37]:
ETAPAS_VALIDAS = ["AI", "AF", "EM"]
COLS_CADASTRO = ["DEP_ADM", "LOCALIZACAO", "REGIAO", "UF"]
TODAS_MAT = [f"QT_MAT_{e}_{t}" for e in ETAPAS_VALIDAS for t in ("REG", "INT")]


def etapas_da_selecao(etapa):
    """Interpreta "TOTAL", "AI", "AF", "EM" ou somas como "AI+AF" (qualquer ordem)."""
    if str(etapa).strip().upper() == "TOTAL":
        return list(ETAPAS_VALIDAS)
    partes = [p.strip().upper() for p in str(etapa).split("+")]
    if any(p not in ETAPAS_VALIDAS for p in partes) or len(set(partes)) != len(partes):
        raise ValueError(f"Etapa inválida: {etapa!r}. Use 'TOTAL', 'AI', 'AF', 'EM' ou somas como 'AI+AF'.")
    return [p for p in ETAPAS_VALIDAS if p in partes]


def colunas_matricula(etapa, integral):
    cols = []
    for e in etapas_da_selecao(etapa):
        cols.append(f"QT_MAT_{e}_REG")
        if integral:
            cols.append(f"QT_MAT_{e}_INT")
    return cols


def adicionar_rede(df):
    """REDE: Federal/Estadual/Municipal -> "Pública"; Privada -> "Privada"."""
    dep = df["DEP_ADM"].astype("string").str.strip().str.lower()
    rede = pd.Series(pd.NA, index=df.index, dtype="object")
    rede[dep.isin(["federal", "estadual", "municipal", "pública", "publica"]).fillna(False)] = "Pública"
    rede[dep.eq("privada").fillna(False)] = "Privada"
    df = df.copy()
    df["REDE"] = rede
    return df


def _limpar(c):
    return str(c).replace("\ufeff", "").replace("ï»¿", "").strip()


def carregar(caminho):
    """Lê só as colunas necessárias, de um .csv ou de um .zip."""
    caminho = Path(caminho)
    precisa = ["ANO", "CO_ESCOLA"] + COLS_CADASTRO + TODAS_MAT
    leitura = dict(sep=CSV_SEP, encoding=CSV_ENCODING, low_memory=False, usecols=lambda c: _limpar(c) in precisa)
    if caminho.suffix.lower() == ".zip":
        with zipfile.ZipFile(caminho) as z:
            validos = [n for n in z.namelist() if n.lower().endswith((".csv", ".txt")) and not n.startswith("__MACOSX")]
            if not validos:
                raise FileNotFoundError("Nenhum .csv encontrado dentro do zip.")
            nome = NOME_ARQUIVO_NO_ZIP or validos[0]
            print("Lendo do zip:", nome)
            with z.open(nome) as f:
                df = pd.read_csv(f, **leitura)
    else:
        print("Lendo:", caminho)
        df = pd.read_csv(caminho, **leitura)
    df.columns = [_limpar(c) for c in df.columns]
    return df


def escola_ano(raw, etapa):
    """Uma linha por escola e ano com MATRICULAS da etapa (soma duplicatas) e filtros aplicados."""
    cols = colunas_matricula(etapa, INCLUIR_INTEGRAL)
    faltando = [c for c in ["ANO", "CO_ESCOLA", DIMENSAO] + cols if c not in raw.columns]
    if faltando:
        raise KeyError(f"Colunas ausentes na base: {faltando}")
    d = raw.copy()
    d["MATRICULAS"] = d[cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
    if d.duplicated(["CO_ESCOLA", "ANO"]).any():
        outras = {c: "first" for c in d.columns if c not in ("CO_ESCOLA", "ANO", "MATRICULAS")}
        d = d.groupby(["CO_ESCOLA", "ANO"], as_index=False).agg({**outras, "MATRICULAS": "sum"})
    for col, filtro in [("REDE", FILTRO_REDE), ("DEP_ADM", FILTRO_DEP_ADM), ("LOCALIZACAO", FILTRO_LOCALIZACAO),
                        ("REGIAO", FILTRO_REGIAO), ("UF", FILTRO_UF)]:
        if filtro is not None:
            d = d[d[col].isin(filtro)]
    d = d.assign(GRUPO=d[DIMENSAO].fillna("(vazio)").astype(str))
    return d[["ANO", "CO_ESCOLA", "GRUPO", "MATRICULAS"]].sort_values(["CO_ESCOLA", "ANO"]).reset_index(drop=True)


def variacao_matricula_total(d):
    """BARRAS: variação % anual da matrícula total de cada grupo (todas as escolas)."""
    tot = d.groupby(["ANO", "GRUPO"])["MATRICULAS"].sum().unstack("GRUPO")
    tot = tot.reindex(range(int(tot.index.min()), int(tot.index.max()) + 1))
    v = tot.pct_change(fill_method=None) * 100
    v = v.loc[(v.index >= ANO_INICIO) & (v.index <= ANO_FIM)]
    v.columns.name = "GRUPO"
    return v.reset_index().melt(id_vars="ANO", var_name="GRUPO", value_name="VAR").dropna()


def pares_consecutivos(d):
    """Uma linha por escola/ano com MAT_ANT, MATRICULAS e VAR_PCT (anos consecutivos, base mínima)."""
    d = d.copy()
    g = d.groupby("CO_ESCOLA")
    d["ANO_ANT"] = g["ANO"].shift(1)
    d["MAT_ANT"] = g["MATRICULAS"].shift(1)
    d = d[(d["ANO"] - d["ANO_ANT"]) == 1]
    d = d[d["MAT_ANT"] >= max(MATRICULA_MINIMA_ANO_ANTERIOR, 1)]
    if EXCLUIR_FECHAMENTO:
        d = d[d["MATRICULAS"] > 0]
    d = d[(d["ANO"] >= ANO_INICIO) & (d["ANO"] <= ANO_FIM)]
    d["VAR_PCT"] = (d["MATRICULAS"] - d["MAT_ANT"]) / d["MAT_ANT"] * 100
    return d.reset_index(drop=True)


def marcar_bruscas(p):
    """Cercas de Tukey (por ano ou global) e flags QUEDA / AUMENTO para cada escola."""
    if CRITERIO not in ("estatistico", "absoluto", "ambos"):
        raise ValueError("CRITERIO deve ser 'estatistico', 'absoluto' ou 'ambos'")
    p = p.copy()
    if LIMITE_PARA_CRUZAMENTO == "ANO":
        q1 = p.groupby("ANO")["VAR_PCT"].transform(lambda s: s.quantile(Q_INFERIOR))
        q3 = p.groupby("ANO")["VAR_PCT"].transform(lambda s: s.quantile(Q_SUPERIOR))
    else:
        q1, q3 = p["VAR_PCT"].quantile(Q_INFERIOR), p["VAR_PCT"].quantile(Q_SUPERIOR)
    aiq = q3 - q1
    est_q = p["VAR_PCT"] < q1 - MULTIPLICADOR * aiq
    est_a = p["VAR_PCT"] > q3 + MULTIPLICADOR * aiq
    abs_q = p["VAR_PCT"] <= CORTE_ABSOLUTO_PCT
    abs_a = p["VAR_PCT"] >= CORTE_ABSOLUTO_AUMENTO_PCT
    if CRITERIO == "estatistico":
        p["QUEDA"], p["AUMENTO"] = est_q, est_a
    elif CRITERIO == "absoluto":
        p["QUEDA"], p["AUMENTO"] = abs_q, abs_a
    else:
        p["QUEDA"], p["AUMENTO"] = est_q & abs_q, est_a & abs_a
    return p


def pct_escolas_bruscas(p):
    """LINHAS: % de escolas de cada grupo e ano com queda brusca e com aumento brusco."""
    partes = []
    for col, tipo in (("QUEDA", "queda"), ("AUMENTO", "aumento")):
        t = p.groupby(["ANO", "GRUPO"]).agg(N=(col, "size"), MARCADAS=(col, "sum")).reset_index()
        t["PCT"] = t["MARCADAS"] / t["N"] * 100
        t["TIPO"] = tipo
        partes.append(t)
    t = pd.concat(partes, ignore_index=True)
    return t[t["N"] >= N_MIN_CATEGORIA]


raw = adicionar_rede(carregar(CAMINHO_ARQUIVO))
print("Base bruta:", raw.shape, "| anos:", int(raw["ANO"].min()), "a", int(raw["ANO"].max()))

barras_etapas, linhas_etapas = [], []
for et in ETAPAS_PAINEIS:
    d = escola_ano(raw, et)
    b = variacao_matricula_total(d)
    b["ETAPA"] = et
    barras_etapas.append(b)
    p = marcar_bruscas(pares_consecutivos(d))
    l = pct_escolas_bruscas(p)
    l["ETAPA"] = et
    linhas_etapas.append(l)
    print(f"{et}: {len(p):,} pares escola-ano usados nas linhas")

barras = pd.concat(barras_etapas, ignore_index=True)
linhas = pd.concat(linhas_etapas, ignore_index=True)

print("\nBarras: variação anual (%) da matrícula total")
display(barras.pivot_table(index="ANO", columns=["ETAPA", "GRUPO"], values="VAR").round(2))
print("Linhas: % de escolas com queda / aumento brusco")
display(linhas.pivot_table(index="ANO", columns=["ETAPA", "GRUPO", "TIPO"], values="PCT").round(1))

Lendo do zip: bd5.csv
Base bruta: (2051631, 13) | anos: 2011 a 2025
AI: 1,120,290 pares escola-ano usados nas linhas
AF: 756,454 pares escola-ano usados nas linhas
EM: 370,684 pares escola-ano usados nas linhas

Barras: variação anual (%) da matrícula total


ETAPA      AF              AI              EM        
GRUPO Privada Pública Privada Pública Privada Pública
ANO                                                  
2012     1.67   -2.79    5.92   -3.47    4.32   -0.92
2013     0.51   -3.29    3.83   -2.56   -0.10   -0.86
2014     0.27   -4.77    3.78   -1.25    0.50   -0.25
2015    -0.46   -3.56    1.15   -2.06   -1.97   -2.82
2016     1.20   -1.33    1.92   -1.32   -3.34    1.31
2017    -1.42   -1.94   -0.33   -0.83   -4.37   -2.22
2018     1.41   -0.36    1.53   -1.53   -3.97   -2.62
2019     1.07   -1.20    0.93   -1.48    0.25   -3.63
2020    -0.03    0.24   -2.24   -1.32   -0.88    1.43
2021    -2.01    0.88   -7.20   -0.49    1.00    3.18
2022     2.31   -1.22    5.42   -0.99    3.89    0.87
2023     3.60   -2.83    1.18   -1.32    1.53   -2.97
2024     2.25   -2.25    2.20    0.09    4.47    1.04
2025    -0.05   -2.22   -0.86    0.34    0.59   -6.30

Linhas: % de escolas com queda / aumento brusco


ETAPA      AF                          AI                          EM        \
GRUPO Privada       Pública       Privada       Pública       Privada         
TIPO  aumento queda aumento queda aumento queda aumento queda aumento queda   
ANO                                                                           
2012      7.3   1.9     4.3   2.5     7.8   2.0     3.0   3.3     8.4   2.6   
2013      6.7   2.3     4.3   3.0     6.6   2.6     3.5   3.6     6.1   3.1   
2014      6.7   1.5     4.0   2.7     6.7   2.4     3.4   3.1     6.2   3.2   
2015      6.0   2.0     4.1   2.7     6.8   2.9     3.4   2.9     5.8   2.4   
2016      6.3   2.2     3.7   2.5     6.7   3.4     3.1   2.9     3.7   3.3   
2017      5.5   2.7     3.8   2.6     6.1   3.8     3.8   2.9     4.8   2.7   
2018      6.5   2.8     3.8   2.5     6.8   3.3     3.5   3.0     4.4   2.7   
2019      6.5   3.0     3.3   2.8     6.7   3.4     3.3   3.0     5.9   2.3   
2020      6.2   3.9     3.2   2.5     6.1   4.5     2.9   2.7     4.3   3.4   
2021      4.3   4.6     3.5   1.9     4.9   7.7     2.9   1.8     3.4   2.2   
2022      7.3   2.0     3.4   2.3    10.4   1.9     2.8   2.5     4.2   1.7   
2023      8.1   2.0     3.1   2.3     6.8   3.4     3.2   2.8     6.2   2.2   
2024      7.5   2.6     3.1   2.6     6.9   4.4     3.1   2.8     6.8   2.7   
2025      7.0   2.7     3.0   2.6     6.4   5.0     3.5   2.7     6.4   2.0   

ETAPA                
GRUPO Pública        
TIPO  aumento queda  
ANO                  
2012      4.5   1.8  
2013      4.4   1.9  
2014      4.1   1.8  
2015      3.5   1.7  
2016      4.2   1.4  
2017      4.3   1.9  
2018      4.2   1.6  
2019      3.4   1.9  
2020      3.7   2.0  
2021      3.1   1.3  
2022      5.3   1.8  
2023      3.6   2.1  
2024      3.6   2.0  
2025      2.9   3.3

## Etapa 3 — O gráfico

Todo o estilo vem dos parâmetros da Etapa 1. Para trocar cores, fontes, tamanhos ou textos, edite lá e rode só esta célula de novo
(e a seguinte, para salvar).

In [38]:
def rotulo_grupo(g):
    return ROTULOS_GRUPOS.get(g, g)


def rotulo_etapa(e):
    return ROTULOS_ETAPAS.get(e, e)


def montar_grafico(barras, linhas):
    b = barras.assign(GRUPO_R=barras["GRUPO"].map(rotulo_grupo), ETAPA_R=barras["ETAPA"].map(rotulo_etapa))
    l = linhas.assign(GRUPO_R=linhas["GRUPO"].map(rotulo_grupo), ETAPA_R=linhas["ETAPA"].map(rotulo_etapa),
                      TIPO_R=linhas["TIPO"].map({"queda": ROTULO_QUEDA, "aumento": ROTULO_AUMENTO}))

    grupos = sorted(set(barras["GRUPO"]) | set(linhas["GRUPO"]))
    if all(g in COR_GRUPOS for g in grupos):
        escala_cor = alt.Scale(domain=[rotulo_grupo(g) for g in grupos], range=[COR_GRUPOS[g] for g in grupos])
    else:
        escala_cor = alt.Scale(scheme=ESQUEMA_CORES)

    # --- escalas dos dois eixos Y ---
    ymax_b = float(np.ceil(barras["VAR"].abs().max()))
    dom_b = [-ymax_b, ymax_b] if Y_BARRAS is None else [float(Y_BARRAS[0]), float(Y_BARRAS[1])]
    ymax_l = float(np.ceil(linhas["PCT"].max()))
    dom_l = [0.0, ymax_l] if Y_LINHAS is None else [float(Y_LINHAS[0]), float(Y_LINHAS[1])]
    if ALINHAR_ZEROS:
        dom_l = [-dom_l[1], dom_l[1]]
    rotulo_dir = "datum.value < 0 ? '' : datum.label" if ALINHAR_ZEROS else None

    paineis = []
    etapas = [e for e in ETAPAS_PAINEIS if e in set(barras["ETAPA"])]
    for et in etapas:
        eb = b[b["ETAPA"] == et]
        el = l[l["ETAPA"] == et]
        ultimo = et == etapas[-1]
        esc_b = alt.Scale(domain=dom_b if EIXO_Y_COMPARTILHADO else [-float(np.ceil(eb["VAR"].abs().max())),
                                                                      float(np.ceil(eb["VAR"].abs().max()))])
        max_l = float(np.ceil(el["PCT"].max()))
        esc_l = alt.Scale(domain=dom_l if EIXO_Y_COMPARTILHADO else ([-max_l, max_l] if ALINHAR_ZEROS else [0.0, max_l]))

        eixo_x = alt.Axis(labelAngle=0, grid=GRADE_ANOS, gridDash=TRACO_GRADE_ANOS, gridColor=COR_GRADE_ANOS,
                          gridWidth=1, tickBand="extent")

        barras_c = alt.Chart(eb).mark_bar(opacity=OPACIDADE_BARRAS).encode(
            x=alt.X("ANO:O", title=TEXTO_EIXO_X if ultimo else None, axis=eixo_x),
            xOffset=alt.XOffset("GRUPO_R:N"),
            y=alt.Y("VAR:Q", title=TEXTO_EIXO_BARRAS, scale=esc_b, axis=alt.Axis(grid=True)),
            color=alt.Color("GRUPO_R:N", title=TEXTO_LEGENDA_GRUPO, scale=escala_cor),
            tooltip=[alt.Tooltip("ANO:O", title="Year"), alt.Tooltip("GRUPO_R:N", title=TEXTO_LEGENDA_GRUPO),
                     alt.Tooltip("VAR:Q", format=".2f", title="Enrollment change (%)")])
        zero = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(color=COR_ZERO).encode(y=alt.Y("y:Q", scale=esc_b, axis=None))

        eixo_dir = alt.Axis(orient="right", grid=False, title=TEXTO_EIXO_LINHAS)
        if rotulo_dir:
            eixo_dir = alt.Axis(orient="right", grid=False, title=TEXTO_EIXO_LINHAS, labelExpr=rotulo_dir)
        linhas_c = alt.Chart(el).mark_line(point=alt.OverlayMarkDef(size=TAMANHO_PONTO, filled=True),
                                           strokeWidth=LARGURA_LINHA).encode(
            x=alt.X("ANO:O"),
            y=alt.Y("PCT:Q", scale=esc_l, axis=eixo_dir),
            color=alt.Color("GRUPO_R:N", title=TEXTO_LEGENDA_GRUPO, scale=escala_cor),
            strokeDash=alt.StrokeDash("TIPO_R:N", title=TEXTO_LEGENDA_TIPO,
                                      scale=alt.Scale(domain=[ROTULO_AUMENTO, ROTULO_QUEDA],
                                                      range=[[1, 0], TRACEJADO_QUEDA]),
                                      legend=alt.Legend(symbolType="stroke", symbolStrokeColor="#374151",
                                                         symbolStrokeWidth=2.5, symbolSize=900,
                                                         symbolFillColor="transparent")),
            tooltip=[alt.Tooltip("ANO:O", title="Year"), alt.Tooltip("GRUPO_R:N", title=TEXTO_LEGENDA_GRUPO),
                     alt.Tooltip("TIPO_R:N", title="Type"), alt.Tooltip("N:Q", title="Schools"),
                     alt.Tooltip("MARCADAS:Q", title="Flagged"), alt.Tooltip("PCT:Q", format=".2f", title="% of schools")])

        camadas = [barras_c, zero, linhas_c] if LINHAS_SOBRE_BARRAS else [linhas_c, barras_c, zero]
        anos_faixa = [a for a in ANOS_PANDEMIA if a in set(eb["ANO"])]
        if anos_faixa:
            faixa = (alt.Chart(pd.DataFrame({"ANO": anos_faixa}))
                     .mark_bar(color=COR_FAIXA, opacity=OPACIDADE_FAIXA, width=alt.RelativeBandSize(1))
                     .encode(x=alt.X("ANO:O"), y=alt.value(0), y2=alt.value(ALTURA_PAINEL)))
            camadas = [faixa] + camadas
        painel = (alt.layer(*camadas).resolve_scale(y="independent")
                  .properties(width=LARGURA_GRAFICO, height=ALTURA_PAINEL,
                              title=alt.TitleParams(rotulo_etapa(et), fontSize=FONTE_TITULO_PAINEL, anchor="start",
                                                    color=COR_TEXTO)))
        paineis.append(painel)

    titulo = alt.TitleParams(TITULO, fontSize=FONTE_TITULO, anchor="start", color=COR_TEXTO,
                             **({"subtitle": SUBTITULO, "subtitleFontSize": FONTE_SUBTITULO,
                                 "subtitleColor": "#6B7280"} if SUBTITULO else {}))
    graf = alt.vconcat(*paineis, spacing=ESPACO_PAINEIS).properties(title=titulo, background=COR_FUNDO)
    return (graf
            .configure_view(stroke=None)
            .configure_axis(gridColor=COR_GRADE_Y, domainColor="#9CA3AF", tickColor="#9CA3AF",
                            labelFontSize=FONTE_EIXOS, titleFontSize=FONTE_EIXOS_TITULO,
                            labelColor=COR_TEXTO_SUAVE, titleColor=COR_TEXTO)
            .configure_legend(labelFontSize=FONTE_LEGENDA, titleFontSize=FONTE_LEGENDA_TITULO, symbolSize=160,
                              labelLimit=400, orient=POSICAO_LEGENDA, labelColor=COR_TEXTO_SUAVE, titleColor=COR_TEXTO))


grafico = montar_grafico(barras, linhas)
display(grafico)


alt.VConcatChart(...)

## Etapa 4 — Salvar a imagem (opcional)

In [39]:
if SALVAR_IMAGEM:
    try:
        PASTA_IMAGENS.mkdir(parents=True, exist_ok=True)
        for fmt in FORMATOS_IMAGEM:
            destino = PASTA_IMAGENS / f"{NOME_IMAGEM}.{fmt}"
            if fmt == "png":
                grafico.save(str(destino), scale_factor=ESCALA_PNG)
            else:
                grafico.save(str(destino))
            print("Salvo em", destino)
    except Exception as e:
        print("[aviso] não foi possível salvar a imagem:", e,
              "\nInstale com: %pip install vl-convert-python  (e reinicie o kernel)")
else:
    print("SALVAR_IMAGEM = False: nada salvo.")

Salvo em c:\Users\niela\Documents\Trabalho Final - Inglês 2\analise\resultados\grafico_combinado.pdf
